In [3]:
# Структура сверточной сети VGG-16
# Количество фильтров увеличивается с каждым слоем, также, как и кол-во Conv
# Вход (3, 224, 244)
# Conv1-1, Conv1-2 (64, 224, 224)
# MaxPooling(2)
# Conv2-1, Conv2-2 (128, 112, 112)
# MaxPoolnig(2)
# Conv3-1, Conv3-2, Conv3-3 (256, 56, 56)
# MaxPooling(2)
# Conv3-1, Conv3-2, Conv3-3 (512, 28, 28)
# MaxPooling(2)
# Conv3-1, Conv3-2, Conv3-3 (512, 14, 14)
# MaxPooling(2)
# Conv3-1, Conv3-2, Conv3-3 (512, 7, 7)
# MaxPooling(2)
# Полносвязная сеть 4096 -> ReLU -> 4096 -> ReLU -> 1000 (образов) -> Softmax

# Два подряд идущих сверточных слоев с фильтром 3x3 эквивалентен одному слою
# с фильтром 5x5, но при этом число настраиваемых параметров в первом случае
# меньше: 5x5 = 25 + 1 = 26, 2x(3x3 + 1) = 20 => меньше вычислений

In [2]:
from torchvision import models

model = models.vgg16() # Необученная модель
print(model)

model = models.vgg16(weights=models.VGG16_Weights.DEFAULT)

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

100.0%


In [ ]:
vgg_weights = models.VGG16_Weights.DEFAULT
transforms = vgg_weights.transforms()

print(vgg_weights.transforms) 

functools.partial(<class 'torchvision.transforms._presets.ImageClassification'>, crop_size=224)


In [8]:
from PIL import Image

img = Image.open('datasets/cnn/car.jpg').convert('RGB')
img_net = transforms(img).unsqueeze(0)

model.eval()
cats = vgg_weights.meta['categories']
p = model(img_net).squeeze() # Результат без выходной функции активации
res = p.softmax(dim=0).sort(descending=True) # Вероятность принадлежности к определенному классу

for s, i in zip(res[0][:5], res[1][:5]):
    print(f'{cats[i]}: {s:.4f}')

beach wagon: 0.5425
sports car: 0.1590
car wheel: 0.0947
convertible: 0.0932
grille: 0.0513


In [ ]:
# unsqueeze(0) добавляет в тензор ось (размерность) соотв. батчу (батч=1), 
# т.к. модель ожидает тензор размерностью (batch, channel, H, W)
# squeeze() удаляет единичные оси, т.е. из (batch, channel, H, W) -> (channel, H, W)